In [77]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [95]:
import pandas as pd
from preprocess import process_description, process_shop_category_name
import polars as pl
from sentence_transformers import SentenceTransformer
import torch
from sklearn.cluster import KMeans
import html
import re
import numpy as np

In [99]:
df = pd.read_csv("train.tsv", sep="\t")

In [100]:
df = process_description(df)

In [7]:
model = SentenceTransformer("cointegrated/rubert-tiny2")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 480.25it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
embeddings = model.encode(df["shop_category_name"].to_list(), convert_to_tensor=True)

In [9]:
embeddings_copy = embeddings.clone()
no_cat_name_mask = (df["shop_category_name"] == "-").to_numpy().astype(bool)
embeddings_copy[no_cat_name_mask] = 0

In [13]:
kmeans = KMeans(16)
clusters = kmeans.fit_predict(embeddings_copy.cpu().numpy())

In [14]:
top_cats = df["shop_category_name"].value_counts().head(21).index.to_list()

In [15]:
df = process_shop_category_name(df, top_cats, model, kmeans)